In [1]:
import ase
import numpy as np
import pyscf
import time
import os
import ase.io
import equiv_dens.utils.base as utils
%load_ext autoreload
%autoreload 2

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


Use "numpy" for Fourier Transform


In [2]:
from pyscf import gto, dft
from pyscf.dft import numint
from pyscf.scf import hf
from pyscf.data import nist

hf.MUTE_CHKFILE = True

#mol = gto.M(atom='O  0  0  0.1184; H  0,  0.7532, -0.4735; H 0,  -0.7532, -0.4735 ', basis='def2svp')
#mf = dft.RKS(mol)
#mf.chkfile=False
#mf.xc = 'pbe'
#mf.kernel()
#g = mf.nuc_grad_method()
#g.kernel()
filename = 'ethanol_10_rot'
mols = list(ase.io.iread(os.path.join('datasets', filename + '.xyz')))
data = {}
data['positions'] = utils.ase_to_npy(mols)
atom_types = mols[0].get_chemical_symbols()
data['atom_types'] = atom_types
data['atom_numbers'] = np.array(utils.symbols_to_numbers(atom_types))
print('atom_numbers', data['atom_numbers'])

results = []
print('results len', len(results))
for i in range(len(results), len(data['positions'])):
    print('calc', i)
    start = time.time()
    pos = data['positions'][i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis='ccpvtz')
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    mf.grids.level = 3
    mf.kernel()
    g = mf.nuc_grad_method()
    forces = g.grad()
    dm = mf.make_rdm1(mf.mo_coeff, mf.mo_occ)
    dpm = mf.dip_moment(mol, dm)
    ao = numint.eval_ao(mol, mf.grids.coords)
                # print('ao time', time.time() - ao_start)
                # rho_start = time.time()
    rho = numint.eval_rho2(mol, ao, mo_coeff=mf.mo_coeff, mo_occ=mf.mo_occ)
    print('density integral:', np.sum(rho * mf.grids.weights))
    print('density shape', rho.shape)
    max_rho = np.argmax(rho)
    print('max rho idx', max_rho)
    print('max rho coords', mf.grids.coords[max_rho, :])
    print('positions', pos)
    print('atom coords', mol.atom_coords())
    grid_e_dpm = np.sum((rho * mf.grids.weights)[:, None] * mf.grids.coords, axis=0)
    grid_nuc_dpm = np.sum(mol.atom_coords() * mol.atom_charges()[:, None], axis=0)
    grid_dpm = grid_nuc_dpm - grid_e_dpm
    grid_dpm *= nist.AU2DEBYE
    print('grids level', mf.grids.level)
    print('grids shape', mf.grids.coords.shape)
    print('grid coords', mf.grids.coords)
    print('grid weights', mf.grids.weights)
    print('rho', rho)
    print('coeff shape', mf.mo_coeff.shape)
    print('occ shape', mf.mo_occ.shape)
    print('dm shape', dm.shape)
    print('elapsed', time.time() - start)
    print('Grid Dipole Moment', grid_dpm)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = forces
    calc_dict['dm'] = dm
    calc_dict['dpm'] = dpm
    calc_dict['grid_dpm'] = grid_dpm
    res.append(calc_dict)
    results.append(res)

atom_numbers [6 6 8 1 1 1 1 1 1]
results len 0
calc 0


KeyboardInterrupt: 

In [6]:
np.save(os.path.join('datasets', filename + '_pyscf.npy'), results, allow_pickle=True)

In [7]:
np.save(os.path.join('datasets', filename + '.npy'), data)